In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

In [3]:
# Displaying the scrapped data in a data frame

goals_list = []
count_goal = 0
n_season = 2025
n_match = 0

for n_round in range(1,39):
    url = f"https://www.transfermarkt.com.br/premier-league/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}"
    response = requests.get(url, headers=headers)
    response.status_code
    soup = BeautifulSoup(response.content, "html.parser")

    all_matches = soup.find_all('table', {'style':'border-top: 0 !important;'})

    season_id = f'PL-{n_season}'

    print(f'{n_round}, ', end="")
    
    for match in all_matches:
        n_match += 1
        match_id = f'M-{n_season}-{n_match}'
        event = match.find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

        # List with the entire class necessary to get the home and away team's names
        gross_h_team = match.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
        gross_a_team = match.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

        # Checking for a possible forum buttom
        home_forum_check = gross_h_team.find('a').get('href')
        away_forum_check = gross_a_team.find('a').get('href')

        # Different ways to get the title depending if it has the forum buttom
        if 'forum' in home_forum_check and 'forum' in away_forum_check:
            h_team = gross_h_team.find_all('a')[1].get('title')
            a_team = gross_a_team.find_all('a')[1].get('title')
        elif 'forum' in home_forum_check:
            h_team = gross_h_team.find_all('a')[1].get('title')
            a_team = gross_a_team.find('a').get('title')
        elif 'forum' in away_forum_check:
            h_team = gross_h_team.find('a').get('title')
            a_team = gross_a_team.find_all('a')[1].get('title')
        else:
            h_team = gross_h_team.find('a').get('title')
            a_team = gross_a_team.find('a').get('title')

        
        for row in event:
            temp = []

            temp.append(season_id)
            temp.append(match_id)

            # Goal primary key
            count_goal += 1
            goal_id = f"G-{count_goal:04d}"
            temp.append(goal_id)

            # Home Team Events
            try: 
                event_type = row.find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
                goal_minute = row.find('td', {'class':'zentriert no-border-links'}).string
                temp.append(h_team)
                temp.append(goal_minute)
            
            # Away Team Events
            except: 
                event_type = row.find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]
                goal_minute = row.find('td', {'class':'zentriert no-border-rechts'}).string
                temp.append(a_team)
                temp.append(goal_minute)

            # Event Types Information
            if event_type == 'icon-tor-formation': temp.append(0) # Normal Goal
            elif event_type == 'icon-elfmeter-formation': temp.append(1) # Penalty Goal
            elif event_type == 'icon-eigentor-formation': temp.append(2) # Own Goal
            elif event_type == 'icon-verschossener-elfmeter-formation': temp.append(-2) # Penalty Missed
            else: temp.append(-1) # Red Cards

            # Player wich made the action
            player = row.find('a').get('title')
            temp.append(player)    

            goals_list.append(temp)

df_goals = pd.DataFrame(goals_list)
df_goals.columns = ['season_id', 'match_id', 'goal_id','goal_score_team','goal_minute','goal_type', 'goal_scorer_name']

display(df_goals.head(10))
display(df_goals.tail(10))

1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 

,season_id,match_id,goal_id,goal_score_team,goal_minute,goal_type,goal_scorer_name
0,PL-2025,M-2025-1,G-0001,FC Liverpool,37',0,Hugo Ekitiké
1,PL-2025,M-2025-1,G-0002,FC Liverpool,49',0,Cody Gakpo
2,PL-2025,M-2025-1,G-0003,AFC Bournemouth,64',0,Antoine Semenyo
3,PL-2025,M-2025-1,G-0004,AFC Bournemouth,76',0,Antoine Semenyo
4,PL-2025,M-2025-1,G-0005,FC Liverpool,88',0,Federico Chiesa
5,PL-2025,M-2025-1,G-0006,FC Liverpool,90+4',0,Mohamed Salah
6,PL-2025,M-2025-2,G-0007,Aston Villa FC,66',-1,Ezri Konsa
7,PL-2025,M-2025-3,G-0008,Brighton & Hove Albion,55',1,Matt O'Riley
8,PL-2025,M-2025-3,G-0009,FC Fulham,90+6',0,Rodrigo Muniz
9,PL-2025,M-2025-4,G-0010,AFC Sunderland,61',0,Eliezer Mayenda


,season_id,match_id,goal_id,goal_score_team,goal_minute,goal_type,goal_scorer_name
1094,PL-2025,M-2025-377,G-1095,Nottingham Forest,34',0,Morgan Gibbs-White
1095,PL-2025,M-2025-377,G-1096,AFC Bournemouth,54',0,Marcus Tavernier
1096,PL-2025,M-2025-378,G-1097,AFC Sunderland,25',0,Trai Hume
1097,PL-2025,M-2025-378,G-1098,AFC Sunderland,50',2,Malo Gusto
1098,PL-2025,M-2025-378,G-1099,Chelsea FC,56',0,Cole Palmer
1099,PL-2025,M-2025-378,G-1100,Chelsea FC,62',-1,Wesley Fofana
1100,PL-2025,M-2025-379,G-1101,Tottenham Hotspur,43',0,João Palhinha
1101,PL-2025,M-2025-380,G-1102,West Ham United,67',0,Taty Castellanos
1102,PL-2025,M-2025-380,G-1103,West Ham United,79',0,Jarrod Bowen
1103,PL-2025,M-2025-380,G-1104,West Ham United,90+4',0,Callum Wilson
